# 4장. 앞의 글자 참고하기: 셀프 어텐션

각 글자가 앞에 나온 글자들을 **얼마나** 참고할지 계산하는 어텐션 헤드를 만들어요. 이 장은 **GPU가 필요 없어요.** (런타임 유형: CPU)

이 장에서 PyTorch의 **`nn.Linear`**, **행렬 곱셈 `@`**, **`softmax`**를 만납니다.


### 준비
3장의 결과물 `Embeddings`를 패키지에서 불러옵니다. 보기 쉽게 벡터 길이를 16으로 작게 잡아요.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab, GPTConfig, Embeddings, show_weights

torch.manual_seed(42)
vocab = Vocab.build()
cfg = GPTConfig(vocab_size=len(vocab), block_size=16, n_embd=16, n_head=1)
emb = Embeddings(cfg)

sentence = "토끼가 숲에 가요"
labels = list(sentence.replace(" ", "␣"))   # 띄어쓰기를 눈에 보이게
idx = torch.tensor(vocab.encode(sentence))
x = emb(idx)
print("글자 수:", len(labels))
print("x 모양:", x.shape)


## 4.1 앞 글자 하나로는 부족하다

3장 놀이터에서 "토" 다음 글자를 세어 봤어요. 앞 글자 하나만 보면 "토끼"인지 "토리"인지 알 수 없었죠.
다음 글자를 잘 맞히려면 **앞 문맥 전체**를 봐야 합니다. 그런데 앞 글자들이 모두 똑같이 중요하지는 않아요.
어텐션은 **각 글자가 앞 글자들을 얼마나씩 참고할지 비율을 계산**하는 장치입니다.


## 4.2 묻고, 찾고, 가져오기: Query, Key, Value

글자마다 벡터 세 개를 만듭니다.

- **Query(질문 쪽지)**: "나는 어떤 정보가 필요해?"
- **Key(명찰)**: "나는 이런 정보를 갖고 있어"
- **Value(메모)**: "내가 실제로 건네줄 정보"

`nn.Linear`는 숫자들에 비중(가중치)을 곱해 더해서 새 숫자들을 만드는 부품이에요.

**핵심 코드**: `q = query(x)`


In [ ]:
head_size = 8
query = nn.Linear(cfg.n_embd, head_size, bias=False)
key = nn.Linear(cfg.n_embd, head_size, bias=False)
value = nn.Linear(cfg.n_embd, head_size, bias=False)
print("query의 가중치 모양:", query.weight.shape)

q = query(x)
k = key(x)
v = value(x)
print("q:", q.shape)
print("k:", k.shape)
print("v:", v.shape)


## 4.3 관련도 점수와 softmax

질문 쪽지와 명찰이 얼마나 잘 맞는지는 **내적**(같은 자리끼리 곱해서 모두 더하기)으로 잽니다. 작은 벡터로 먼저 해 봅시다.

**핵심 코드**: `scores = q @ kt`


In [ ]:
q1 = torch.tensor([1.0, 2.0, 0.0])
key_a = torch.tensor([2.0, 1.0, 0.0])
key_b = torch.tensor([0.0, 0.0, 3.0])
key_c = torch.tensor([-1.0, -2.0, 0.0])

print("같은 자리끼리 곱하기:", q1 * key_a)
print("명찰 A 점수:", (q1 * key_a).sum())
print("명찰 B 점수:", (q1 * key_b).sum())
print("명찰 C 점수:", (q1 * key_c).sum())


질문 쪽지와 명찰을 줄로 쌓아 표로 만들면, **행렬 곱셈 `@`** 한 번으로 모든 점수를 계산할 수 있어요. 첫 줄이 위에서 하나씩 계산한 4, 0, -5와 같은지 보세요.


In [ ]:
Q = torch.tensor([[1.0, 2.0, 0.0],         # 4-3의 질문 쪽지 q1
                  [0.0, 1.0, 1.0]])        # 질문 쪽지 하나 더
K = torch.tensor([[2.0, 1.0, 0.0],         # 명찰 A
                  [0.0, 0.0, 3.0],         # 명찰 B
                  [-1.0, -2.0, 0.0]])      # 명찰 C
print(Q @ K.T)


4.2의 `nn.Linear`도 사실 행렬 곱셈이에요. 레시피 표(`query.weight`)를 뒤집어 `x`와 곱하면 `query(x)`와 같은 결과가 나옵니다.


In [ ]:
by_hand = x @ query.weight.T             # (9, 16) @ (16, 8) → (9, 8)
print("직접 계산한 모양:", by_hand.shape)
print("query(x)와 같은가요?", torch.allclose(by_hand, q))


이제 진짜 글자 9개의 질문 쪽지와 명찰로 모든 쌍의 점수를 계산합니다.


In [ ]:
kt = k.transpose(-2, -1)
print("k:", k.shape, "→ 뒤집은 k:", kt.shape)

scores = q @ kt
scores = scores / head_size ** 0.5
print("점수 표 모양:", scores.shape)


**softmax**는 점수를 모두 0보다 크고 합이 1인 **비율**로 바꿔요. 점수 세 개로 먼저 확인합니다.


In [ ]:
s = torch.tensor([2.0, 1.0, 0.0])
e = torch.exp(s)
print("지수 함수로 키우기:", e)
print("합으로 나누기:", e / e.sum())
print("softmax:", F.softmax(s, dim=-1))


In [ ]:
weights = F.softmax(scores, dim=-1)
show_weights(weights, labels)


## 4.4 미래를 가리는 마스크

위 표에서는 첫 글자 "토"가 뒤에 올 글자까지 참고하고 있어요. 다음 글자를 맞히는 문제에서 **뒤 글자를 보는 것은 정답을 미리 보는 것**입니다.
아래 삼각형 마스크로 뒤 글자의 점수를 -∞로 바꾸면, softmax 뒤 그 비율이 0이 됩니다.

**핵심 코드**: `scores.masked_fill(mask == 0, float(\"-inf\"))`


In [ ]:
T = len(labels)
mask = torch.tril(torch.ones(T, T))
print(mask)


In [ ]:
masked = scores.masked_fill(mask == 0, float("-inf"))
weights = F.softmax(masked, dim=-1)
show_weights(weights, labels)


## 4.5 어텐션 헤드 완성하기

마지막으로 비율대로 Value(메모)를 섞습니다(가중합). 각 글자는 앞 글자들의 정보를 비율만큼 모은 새 벡터가 돼요.
그다음 지금까지의 과정을 `Head` 클래스 하나로 묶습니다.

**핵심 코드**: `out = weights @ v` — 비율대로 메모 섞기


In [ ]:
out = weights @ v
print("비율 표:", weights.shape, "@ 메모:", v.shape)
print("결과:", out.shape)


| 단계 | 모양 | 줄은 | 칸은 | 앞에 쌓인 것 |
|---|---|---|---|---|
| x | (9, 16) | 글자 9개 | 숫자 16개 | — |
| q, k, v | (9, 8) | 글자 9개 | 숫자 8개 | — |
| k 뒤집기 | (8, 9) | 숫자 8개 | 글자 9개 | — |
| 점수, 비율 | (9, 9) | 지금 글자 | 참고하는 글자 | — |
| 결과 | (9, 8) | 글자 9개 | 숫자 8개 | — |


In [ ]:
class Head(nn.Module):
    def __init__(self, cfg, head_size):
        super().__init__()
        self.query = nn.Linear(cfg.n_embd, head_size, bias=False)
        self.key = nn.Linear(cfg.n_embd, head_size, bias=False)
        self.value = nn.Linear(cfg.n_embd, head_size, bias=False)

    def forward(self, x):
        T = x.shape[-2]
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1)                  # ① 관련도
        scores = scores / k.shape[-1] ** 0.5
        mask = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(mask == 0, float("-inf"))  # ② 미래 가리기
        weights = F.softmax(scores, dim=-1)               # ③ 비율로 바꾸기
        self.weights = weights                            # 비율 표를 남겨 두기
        out = weights @ v                                 # ④ 가중합
        return out

head = Head(cfg, head_size)
out = head(x)
print("입력:", x.shape)
print("출력:", out.shape)
show_weights(head.weights, labels)


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

마스크를 빼면 비율 표가 어떻게 달라지는지 보세요. 오른쪽 위 칸에도 값이 생기면, 모델이 정답을 미리 볼 수 있다는 뜻이에요.


In [ ]:
q = head.query(x)
k = head.key(x)
scores = q @ k.transpose(-2, -1) / head_size ** 0.5
w_no_mask = F.softmax(scores, dim=-1)
show_weights(w_no_mask, labels)


## 실제 LLM에서는

- 실제 LLM은 수천~수십만 토큰 길이의 문맥을 봅니다. 문맥이 길어지면 이 표가 엄청나게 커지므로, 표를 통째로 만들지 않고 조각조각 계산하는 기법(FlashAttention 등)을 써요.
- 생성할 때는 앞에서 계산한 Key와 Value를 저장해 두고 다시 쓰는 기법(KV 캐시)으로 속도를 높입니다.
